# Implement Feed-Forward Network from Scratch
### Problem Statement
The Feed-Forward Network (FFN) is the other main component in each Transformer layer (alongside attention). It's a simple **position-wise** MLP applied independently to each token.

The original Transformer uses a two-layer MLP with ReLU:
```
FFN(x) = W2 · ReLU(W1 · x + b1) + b2
```

Modern LLMs (Llama, Mistral, Gemma) use **SwiGLU**, a gated variant:
```
SwiGLU(x) = W2 · (Swish(W_gate · x) ⊙ (W_up · x))
```
where `Swish(x) = x · sigmoid(x)` and `⊙` is element-wise multiplication.

Your goal is to implement both variants from scratch.

---

### Requirements

1. **Standard FFN (ReLU)**
   - Two linear layers: `d_model → d_ff → d_model`
   - ReLU activation between them
   - Typically `d_ff = 4 * d_model`

2. **SwiGLU FFN**
   - Three linear layers: `W_gate`, `W_up` (both `d_model → d_ff`), `W_down` (`d_ff → d_model`)
   - Gate: `Swish(W_gate(x))` where `Swish(x) = x * sigmoid(x)` (same as `F.silu`)
   - Output: `W_down(Swish(W_gate(x)) * W_up(x))`

3. **Validate**
   - Output shape must be `(batch_size, seq_len, d_model)` for both.
   - Verify parameter counts match expectations.

---

### Constraints

- ✅ Use only PyTorch operations.
- ✅ Both variants must maintain the input/output dimension `d_model`.
- ✅ SwiGLU should use `F.silu` (Swish activation).

---

<details>
  <summary>💡 Hint</summary>

  - Standard FFN: `nn.Linear(d_model, d_ff)` → `F.relu()` → `nn.Linear(d_ff, d_model)`
  - SwiGLU has **3 weight matrices** instead of 2, but `d_ff` is often set to `(2/3) * 4 * d_model` to keep total params similar.
  - `F.silu(x)` is `x * torch.sigmoid(x)` — the Swish activation.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class FeedForward(nn.Module):
    """
    Standard Transformer FFN with ReLU activation.
    FFN(x) = W2 * ReLU(W1 * x + b1) + b2
    """
    def __init__(self, d_model: int, d_ff: int = None, dropout: float = 0.1):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.w1 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w2(self.dropout(F.relu(self.w1(x))))

In [ ]:
class SwiGLUFeedForward(nn.Module):
    """
    SwiGLU Feed-Forward Network used in Llama, Mistral, Gemma.
    SwiGLU(x) = W_down * (Swish(W_gate * x) * W_up * x)
    """
    def __init__(self, d_model: int, d_ff: int = None, dropout: float = 0.1):
        super().__init__()
        # SwiGLU typically uses (2/3) * 4 * d_model for d_ff
        # to keep param count similar to standard FFN (3 matrices of d*d_ff vs 2 matrices of d*4d)
        d_ff = d_ff or int(2 / 3 * 4 * d_model)
        self.w_gate = nn.Linear(d_model, d_ff, bias=False)
        self.w_up = nn.Linear(d_model, d_ff, bias=False)
        self.w_down = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Gate branch: apply Swish activation
        gate = F.silu(self.w_gate(x))  # Swish = x * sigmoid(x)
        # Up branch: linear projection
        up = self.w_up(x)
        # Element-wise product of gate and up, then project down
        return self.w_down(self.dropout(gate * up))

## 📚 FFN Variants Explained

### Standard FFN (Original Transformer, GPT-2/3, BERT)

```
x ──→ [W1: d→4d] ──→ ReLU ──→ [W2: 4d→d] ──→ output
```

- 2 weight matrices: `W1 (d × 4d)` + `W2 (4d × d)` = **8d²** params
- Simple and effective

### SwiGLU FFN (Llama, Mistral, PaLM, Gemma)

```
         ┌→ [W_gate: d→d'] → Swish ──┐
x ──────┤                              ⊙ ──→ [W_down: d'→d] → output
         └→ [W_up:   d→d'] ──────────┘
```

- 3 weight matrices: `W_gate (d × d')` + `W_up (d × d')` + `W_down (d' × d)` = **3dd'** params
- With `d' = (2/3) * 4d`: 3 × d × (8d/3) = **8d²** params (same as standard!)
- The gating mechanism allows the network to selectively pass information

### Why SwiGLU?

1. **Gating**: The element-wise product `Swish(gate) ⊙ up` lets the network learn to selectively activate dimensions
2. **Swish**: Smoother than ReLU, allows small negative values to pass through
3. **Empirically better**: PaLM paper showed SwiGLU consistently outperforms ReLU/GELU FFN

### Activation Functions Comparison

| Activation | Formula | Used in |
|-----------|---------|--------|
| ReLU | `max(0, x)` | Original Transformer |
| GELU | `x * Φ(x)` | BERT, GPT-2 |
| Swish/SiLU | `x * sigmoid(x)` | Llama (inside SwiGLU) |
| SwiGLU | `Swish(Wx) ⊙ Vx` | Llama, Mistral, PaLM |

In [ ]:
# Test both FFN variants
torch.manual_seed(42)
d_model = 64
x = torch.randn(3, 4, d_model)  # (batch, seq_len, d_model)

ffn = FeedForward(d_model, dropout=0.0)
swiglu = SwiGLUFeedForward(d_model, dropout=0.0)

out_ffn = ffn(x)
out_swiglu = swiglu(x)

print(f"Input shape:       {x.shape}")
print(f"FFN output shape:  {out_ffn.shape}")
print(f"SwiGLU output:     {out_swiglu.shape}")
assert out_ffn.shape == x.shape
assert out_swiglu.shape == x.shape

# Compare parameter counts
ffn_params = sum(p.numel() for p in ffn.parameters())
swiglu_params = sum(p.numel() for p in swiglu.parameters())
print(f"\nFFN params:    {ffn_params:,}")
print(f"SwiGLU params: {swiglu_params:,}")
print("\n✅ Both FFN variants work correctly.")

In [ ]:
# Bonus: Verify that F.silu matches manual Swish computation
x_test = torch.randn(5)
swish_manual = x_test * torch.sigmoid(x_test)
swish_pytorch = F.silu(x_test)
assert torch.allclose(swish_manual, swish_pytorch)
print("✅ F.silu == x * sigmoid(x) confirmed")